# 🏋️ Exercise 5 — Dashboard
**Tujuan:** membuat dashboard penjualan dari data di database.

> Isi bagian bertanda `# TODO`, lalu jalankan sel **Cek Jawaban**.

## ⚙️ Persiapan
Pastikan kernel **.venv** sudah dipilih (pojok kanan atas) dan file `.env` sudah diisi, lalu jalankan sel-sel berikut.

In [1]:
# ==== KONFIGURASI ====
# Kredensial database dibaca dari file .env di root repo (salin .env.example → .env, isi dari kartu akses).
import os
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True))

GITHUB_USER = "AisahAlfiyatusR"     # akun GitHub pemateri (repo asal yang Anda fork)
REPO        = "data-science-workshop"
RAW_BASE    = f"https://raw.githubusercontent.com/{GITHUB_USER}/{REPO}/main"
API_URL     = os.getenv("API_URL") or RAW_BASE + "/api/static/sales_page_{page}.json"

DB_HOST     = os.getenv("DB_HOST")
DB_PORT     = int(os.getenv("DB_PORT", 5432))
DB_NAME     = os.getenv("DB_NAME", "workshop")
DB_USER     = os.getenv("DB_USER")
DB_PASSWORD = os.getenv("DB_PASSWORD")
SCHEMA      = DB_USER   # setiap peserta punya schema pribadi bernama sama dengan username

if not (DB_HOST and DB_USER and DB_PASSWORD):
    print("⚠️ File .env belum ditemukan/diisi — bagian database tidak akan berjalan.")
else:
    print(f"Konfigurasi OK — login database sebagai {DB_USER} @ {DB_HOST}")

Konfigurasi OK — login database sebagai peserta25 @ ep-wandering-math-b3bml8w1-pooler.c-4.ap-southeast-1.aws.neon.tech


In [2]:
import os, requests
import pandas as pd
from sqlalchemy import create_engine, text, Date
from sqlalchemy.engine import URL
pd.set_option("display.max_columns", 20)


def page_url(base, page, page_size=100):
    """URL satu halaman — mendukung API statis ('{page}' di URL) maupun API server (?page=)."""
    if "{page}" in base:
        return base.format(page=page)
    return f"{base}?page={page}&page_size={page_size}"

In [3]:
engine = create_engine(URL.create("postgresql+psycopg2", username=DB_USER, password=DB_PASSWORD,
                                  host=DB_HOST, port=DB_PORT, database=DB_NAME),
                       pool_size=1, max_overflow=1, pool_pre_ping=True)   # hemat koneksi: ±70 peserta memakai server yang sama
with engine.connect() as conn:
    print(conn.execute(text("SELECT current_user, current_schema()")).fetchone())

('peserta25', 'peserta25')


In [4]:
def run_sql(query):
    """Menjalankan query SELECT dan mengembalikan DataFrame."""
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)

In [5]:
TABLE = f"{SCHEMA}.sales_clean"   # ganti ke "ref.sales_clean" jika Load Anda belum berhasil

## Bagian A — Dashboard di notebook (Plotly)
Lengkapi query berikut. Masing-masing akan dipakai sebagai sumber satu komponen dashboard.

In [6]:
# KPI utama
kpi = run_sql(f"""
    SELECT
        SUM(total_amount) AS revenue,
        COUNT(*) AS transaksi,
        COUNT(DISTINCT customer_id) AS pelanggan,
        AVG(total_amount) AS aov
    FROM {TABLE}
""").iloc[0]

# Revenue per tanggal
harian = run_sql(f"""
    SELECT
        transaction_date::date AS transaction_date,
        SUM(total_amount) AS revenue
    FROM {TABLE}
    GROUP BY transaction_date::date
    ORDER BY transaction_date::date
""")

# Revenue per kota
per_kota = run_sql(f"""
    SELECT
        city,
        SUM(total_amount) AS revenue
    FROM {TABLE}
    GROUP BY city
    ORDER BY revenue DESC
""")

# Revenue per kategori
per_kat = run_sql(f"""
    SELECT
        category,
        SUM(total_amount) AS revenue
    FROM {TABLE}
    GROUP BY category
    ORDER BY revenue DESC
""")

In [7]:
assert float(kpi.revenue) == 146_350_000 and int(kpi.transaksi) == 300 and int(kpi.pelanggan) == 59, "Nilai KPI belum tepat"
assert len(harian) > 80 and len(per_kota) == 5 and len(per_kat) == 3, "Query belum lengkap — periksa kembali TODO"
print("✅ Data dashboard benar — lanjut buat visual")

✅ Data dashboard benar — lanjut buat visual


Jalankan sel berikut untuk merender dashboard (boleh dimodifikasi warna/judulnya).

In [10]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

rp = lambda x: f"Rp {x/1e6:,.1f} jt"
fig = make_subplots(
    rows=3, cols=4,
    specs=[[{"type": "indicator"}] * 4,
           [{"type": "xy", "colspan": 4}, None, None, None],
           [{"type": "xy", "colspan": 2}, None, {"type": "domain", "colspan": 2}, None]],
    row_heights=[0.18, 0.42, 0.40], vertical_spacing=0.10,
    subplot_titles=("", "", "", "", "Revenue Harian", "Revenue per Kota", "Komposisi Kategori"))

for i, (label, val, fmt) in enumerate([("Total Revenue", kpi.revenue / 1e6, ",.1f"),
                                       ("Transaksi", kpi.transaksi, ","),
                                       ("Pelanggan", kpi.pelanggan, ","),
                                       ("Rata-rata/Transaksi", kpi.aov / 1e3, ",.0f")], start=1):
    suffix = " jt" if i == 1 else (" rb" if i == 4 else "")
    fig.add_trace(go.Indicator(mode="number", value=float(val), title={"text": label},
                               number={"valueformat": fmt, "suffix": suffix,
                                       "prefix": "Rp " if i in (1, 4) else ""}), row=1, col=i)

fig.add_trace(go.Scatter(x=harian.transaction_date, y=harian.revenue, mode="lines",
                         line=dict(color="#710F76"), name="Revenue"), row=2, col=1)
fig.add_trace(go.Bar(x=per_kota.city, y=per_kota.revenue, marker_color="#3F0951",
                     text=[rp(v) for v in per_kota.revenue], textposition="outside"), row=3, col=1)
fig.add_trace(go.Pie(labels=per_kat.category, values=per_kat.revenue, hole=0.5,
                     marker=dict(colors=["#C343D6", "#4E194F", "#E1B6E7"])), row=3, col=3)
fig.update_layout(height=900, showlegend=False, title="Dashboard Penjualan Q1 2026",
                  template="plotly_white", margin=dict(t=80, l=40, r=40, b=40))
fig.show()

## Bagian B — Dashboard di Looker Studio
1. Buka https://lookerstudio.google.com → **Create → Report**
2. Pilih konektor **PostgreSQL** → isi Host, Port, Database, Username, Password (dari kartu akses)
3. Pilih tabel `sales_clean` di schema Anda → **Add**
4. Bangun komponen sesuai template (`09_dashboard_template/README.md`):
   - 4 Scorecard: Revenue, Transaksi, Pelanggan (Count Distinct), Rata-rata transaksi
   - Time series revenue harian · Bar chart revenue per kota · Donut kategori · Tabel top produk
   - Kontrol filter: rentang tanggal, kota, kategori
5. **Share** → salin link, tempel di bawah.

🔗 Link dashboard saya: ...